# Bar Path Model

### Training a model for bar path tracking

In [47]:
import numpy as np
import pandas as pd

## Grab data from AirTable

In [48]:
from ast import literal_eval
from pyairtable import Api as airtable_api

In [95]:
CUTOFF_FREQ = 0.1
SAMPLE_RATE = 100

WINDOW_SIZE = 100
STEP_SIZE = 50

N_ESTIMATORS = 100
MAX_DEPTH = 10  
RANDOM_STATE = 0

AIR_TABLE_API_KEY = 'patMswkrKzfHWSG3U.a882cccc6e7709b3a24b42b7f9c0ccd48ed15614f13c54e4d3dbaa55ba8ffffb'
BASE_ID = 'appaCiWbsAFmlkOLF'
TABLE_NAME = 'tblZtbDVHfCaoTnsH'
BASE_URL = 'https://api.airtable.com/v0/'

EXERCISE = 'Bench'
VALID_DATA = 'true'
MIN_REPS = '1'

FORMULA = "AND({ValidData} = 'true', " + \
        "{Exercise} = '" + EXERCISE + "', " + \
        "{StartRepTime} != '', " + \
        "{EndRepTime} != '', " + \
        "{Reps} >= '" + MIN_REPS + "')"

FORMULA

"AND({ValidData} = 'true', {Exercise} = 'Bench', {StartRepTime} != '', {EndRepTime} != '', {Reps} >= '1')"

In [75]:
def to_numpy(data):
    return np.array(literal_eval(data))

def to_float(data):
    return float(data)

In [76]:
api = airtable_api(AIR_TABLE_API_KEY)
table = api.table(BASE_ID, TABLE_NAME)

matches = table.all(formula=FORMULA)
print("Number of matching records: ", len(matches))

Number of matching records:  41


In [77]:
df = pd.DataFrame(columns=[
    'Date',
    'Exercise',
    'Lifter',
    'WorkoutTime',
    'Reps',
    'Weight',
    'Intensity',
    'Notes',
    'StartRepTime',
    'EndRepTime',
    'Counter',
    'TimeBetweenSamples',
    'AccX',
    'AccY',
    'AccZ',
    'Pitch',
    'Roll',
    'Yaw',
    'HeartRate'
])

data_dict = []

for match in matches:
    data = match['fields']
    
    data_dict.append({
        'Date': data.get('Date'),
        'Exercise': data.get('Exercise'),
        'Lifter': data.get('Lifter'),
        'WorkoutTime': data.get('WorkoutTime'),
        'Reps': data.get('Reps'),
        'Weight': data.get('Weight'),
        'Intensity': data.get('Intensity'),
        'Notes': data.get('Notes'),
        'StartRepTime': data.get('StartRepTime'),
        'EndRepTime': data.get('EndRepTime'),
        'Counter': to_numpy(data.get('Counter')),
        'TimeBetweenSamples': to_numpy(data.get('TimeBetweenSamples')),
        'AccX': to_numpy(data.get('AccX')),
        'AccY': to_numpy(data.get('AccY')),
        'AccZ': to_numpy(data.get('AccZ')),
        'Pitch': to_numpy(data.get('Pitch')),
        'Roll': to_numpy(data.get('Roll')),
        'Yaw': to_numpy(data.get('Yaw')),
        'HeartRate': to_numpy(data.get('HeartRate')),
    })

df = pd.concat([pd.DataFrame([d]) for d in data_dict], ignore_index=True)
df

,Date,Exercise,Lifter,WorkoutTime,Reps,Weight,Intensity,Notes,StartRepTime,EndRepTime,Counter,TimeBetweenSamples,AccX,AccY,AccZ,Pitch,Roll,Yaw,HeartRate
0,"Apr 27, 2024 at 10:41:29 AM",Bench,Anwar,35.35,6,205,9,None,13.44,27.93,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[205284, 8, 11, 9, 10, 10, 10, 12, 9, 11, 9, 1...","[0.057, 0.192, 0.262, 0.207, 0.184, 0.111, -0....","[-0.014, 0.239, 0.342, 0.259, 0.248, 0.047, -0...","[0.073, -0.095, -0.129, -0.078, -0.054, -0.009...","[0.383, 0.383, 0.383, 0.383, 0.383, 0.382, 0.3...","[0.041, 0.04, 0.04, 0.04, 0.04, 0.041, 0.04, 0...","[-0.008, -0.007, -0.007, -0.007, -0.008, -0.00...","[88, 88, 88, 88, 88, 88, 88, 88, 88, 88, 88, 8..."
1,"Mar 6, 2024 at 7:00:55 AM",Bench,Anwar,35.75,4,240,10,None,15.04,28.92,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[185403, 9, 13, 9, 8, 10, 10, 10, 10, 10, 10, ...","[-0.055, -0.163, -0.174, 0.001, 0.017, -0.031,...","[-0.172, -0.25, -0.205, -0.101, -0.065, -0.08,...","[0.077, 0.173, 0.187, 0.127, 0.037, -0.014, 0....","[-0.31, -0.31, -0.311, -0.312, -0.312, -0.313,...","[0.209, 0.21, 0.21, 0.21, 0.211, 0.211, 0.212,...","[0.033, 0.033, 0.033, 0.034, 0.034, 0.035, 0.0...","[99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 9..."
2,"Mar 6, 2024 at 6:53:59 AM",Bench,Anwar,29.18,5,225,8,None,13.53,24.54,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[161517, 29, 0, 0, 9, 10, 10, 10, 10, 10, 10, ...","[0.097, 0.129, 0.04, 0.06, 0.111, 0.056, 0.041...","[0.071, -0.012, -0.164, -0.167, -0.11, -0.126,...","[0.185, 0.168, 0.182, 0.242, 0.282, 0.36, 0.47...","[0.42, 0.419, 0.419, 0.418, 0.417, 0.415, 0.41...","[-0.035, -0.035, -0.036, -0.037, -0.038, -0.03...","[0.008, 0.007, 0.007, 0.007, 0.007, 0.007, 0.0...","[80, 80, 80, 80, 80, 80, 81, 81, 81, 81, 81, 8..."
3,"Mar 6, 2024 at 6:50:48 AM",Bench,Anwar,27.94,5,185,5,None,13.95,23.43,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[167543, 9, 10, 11, 9, 11, 9, 10, 10, 10, 10, ...","[0.032, 0.217, 0.169, -0.01, 0.045, 0.196, 0.1...","[0.273, 0.248, 0.152, 0.091, 0.209, 0.219, 0.1...","[0.284, 0.239, 0.126, 0.027, 0.042, 0.055, 0.0...","[-0.249, -0.25, -0.251, -0.251, -0.251, -0.252...","[0.308, 0.308, 0.307, 0.307, 0.306, 0.305, 0.3...","[0.039, 0.039, 0.039, 0.039, 0.04, 0.04, 0.04,...","[112, 112, 112, 112, 112, 112, 112, 112, 112, ..."
4,"Mar 2, 2024 at 2:50:08 PM",Bench,Anwar,26.90,5,205,7,None,13.00,22.62,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[109533, 10, 10, 10, 10, 10, 10, 10, 10, 10, 1...","[0.163, 0.025, -0.14, -0.131, 0.103, 0.198, 0....","[0.14, 0.243, 0.188, 0.056, 0.03, 0.048, -0.02...","[-0.171, -0.186, -0.241, -0.232, -0.162, -0.08...","[0.611, 0.61, 0.611, 0.611, 0.611, 0.611, 0.61...","[0.001, 0.0, 0.0, -0.0, -0.001, -0.001, -0.002...","[-0.0, 0.001, 0.001, 0.002, 0.003, 0.005, 0.00...","[100, 100, 100, 100, 100, 100, 100, 100, 100, ..."
5,"Jun 4, 2024 at 4:32:13 PM",Bench,Anwar,36.65,6,205,8,None,16.88,30.08,"[0, 0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[340167980, 1, 9, 9, 10, 10, 10, 11, 9, 10, 10...","[0.007, -0.189, -0.047, 0.089, 0.086, 0.005, -...","[-0.306, 0.044, 0.086, -0.049, -0.097, -0.004,...","[0.174, 0.051, 0.003, 0.129, 0.196, 0.27, 0.21...","[0.218, 0.218, 0.218, 0.217, 0.217, 0.216, 0.2...","[-0.017, -0.017, -0.018, -0.018, -0.018, -0.01...","[0.002, 0.002, 0.002, 0.001, 0.001, 0.001, 0.0...","[89, 89, 89, 89, 89, 89, 89, 89, 89, 89, 89, 8..."
6,"Mar 17, 2024 at 1:43:08 PM",Bench,Anwar,36.49,5,225,9,None,17.11,30.2,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[162425, 12, 8, 10, 10, 10, 10, 10, 10, 10, 10...","[0.051, -0.059, -0.075, -0.134, -0.059, -0.055...","[0.323, 0.241, 0.198, -0.062, -0.134, -0.027, ...","[-0.135, -0.048, -0.096, 0.03, 0.099, 0.149, 0...","[0.313, 0.315, 0.317, 0.318, 0.319, 0.319, 0.3...","[0.074, 0.074, 0.073, 0.073, 0.072, 0.072, 0.0...","[-0.012, -0.012, -0.013, -0.013, -0.012, -0.01...","[98, 98, 98, 98, 98, 98, 98, 98, 98, 98, 98, 9..."
7,"Apr 20, 2024 at 10:36:11 AM",Bench,Anwar,50.23,6,215,9

## Filter and integrate

In [78]:
from scipy.integrate import cumulative_trapezoid
from scipy.signal import find_peaks, filtfilt, butter

In [80]:
def filter(data, cutoff_freq, sample_rate):
    b, a = butter(1, 2 * cutoff_freq / (sample_rate), btype="highpass")
    return filtfilt(b, a, data)

In [81]:
df['AccX'] = df['AccX'].apply(lambda x: filter(x, CUTOFF_FREQ, SAMPLE_RATE))
df['AccY'] = df['AccY'].apply(lambda x: filter(x, CUTOFF_FREQ, SAMPLE_RATE))
df['AccZ'] = df['AccZ'].apply(lambda x: filter(x, CUTOFF_FREQ, SAMPLE_RATE))

df['VelX'] = df['AccX'].apply(lambda x: cumulative_trapezoid(x, dx=1/SAMPLE_RATE))
df['VelY'] = df['AccY'].apply(lambda x: cumulative_trapezoid(x, dx=1/SAMPLE_RATE))
df['VelZ'] = df['AccZ'].apply(lambda x: cumulative_trapezoid(x, dx=1/SAMPLE_RATE))

df['PosX'] = df['VelX'].apply(lambda x: cumulative_trapezoid(x, dx=1/SAMPLE_RATE))
df['PosY'] = df['VelY'].apply(lambda x: cumulative_trapezoid(x, dx=1/SAMPLE_RATE))
df['PosZ'] = df['VelZ'].apply(lambda x: cumulative_trapezoid(x, dx=1/SAMPLE_RATE))

df

,Date,Exercise,Lifter,WorkoutTime,Reps,Weight,Intensity,Notes,StartRepTime,EndRepTime,...,Pitch,Roll,Yaw,HeartRate,VelX,VelY,VelZ,PosX,PosY,PosZ
0,"Apr 27, 2024 at 10:41:29 AM",Bench,Anwar,35.35,6,205,9,None,13.44,27.93,...,"[0.383, 0.383, 0.383, 0.383, 0.383, 0.382, 0.3...","[0.041, 0.04, 0.04, 0.04, 0.04, 0.041, 0.04, 0...","[-0.008, -0.007, -0.007, -0.007, -0.008, -0.00...","[88, 88, 88, 88, 88, 88, 88, 88, 88, 88, 88, 8...","[-0.003195888462468312, -0.0053848632470395175...","[-0.0023192858278520134, -0.00286637489803972,...","[-0.002395584962981559, -0.005808193964378528,...","[-4.2903758547539145e-05, -0.00010741317552610...","[-2.592830362945867e-05, -5.68667050713439e-05...","[-4.101889463680043e-05, -0.000115774619075771..."
1,"Mar 6, 2024 at 7:00:55 AM",Bench,Anwar,35.75,4,240,10,None,15.04,28.92,...,"[-0.31, -0.31, -0.311, -0.312, -0.312, -0.313,...","[0.209, 0.21, 0.21, 0.21, 0.211, 0.211, 0.212,...","[0.033, 0.033, 0.033, 0.034, 0.034, 0.035, 0.0...","[99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 9...","[-0.007841385873689807, -0.01632887224661385, ...","[-0.0024064690738754254, -0.00499308504834592,...","[-0.002820559592569354, -0.005108135793820296,...","[-0.00012085129060151828, -0.00032273455299006...","[-3.6997770611106724e-05, -9.621290114838193e-...","[-3.9643476931948246e-05, -0.00010339828896877..."
2,"Mar 6, 2024 at 6:53:59 AM",Bench,Anwar,29.18,5,225,8,None,13.53,24.54,...,"[0.42, 0.419, 0.419, 0.418, 0.417, 0.415, 0.41...","[-0.035, -0.035, -0.036, -0.037, -0.038, -0.03...","[0.008, 0.007, 0.007, 0.007, 0.007, 0.007, 0.0...","[80, 80, 80, 80, 80, 80, 81, 81, 81, 81, 81, 8...","[-0.008654192213433094, -0.01764551417892527, ...","[-0.003445087329499873, -0.008064759736085645,...","[-0.0026576431597704, -0.005364412911090466, -...","[-0.00013149853196179183, -0.00035489771014474...","[-5.754923532792759e-05, -0.000165169012269391...","[-4.011028035430433e-05, -0.000105609406483978..."
3,"Mar 6, 2024 at 6:50:48 AM",Bench,Anwar,27.94,5,185,5,None,13.95,23.43,...,"[-0.249, -0.25, -0.251, -0.251, -0.251, -0.252...","[0.308, 0.308, 0.307, 0.307, 0.306, 0.305, 0.3...","[0.039, 0.039, 0.039, 0.039, 0.04, 0.04, 0.04,...","[112, 112, 112, 112, 112, 112, 112, 112, 112, ...","[-1.9349340328203638e-05, 0.000634104017285472...","[-0.001055613463099274, -0.0027166669876206246...","[-0.0007411546566765374, -0.002260759270630684...","[3.073773384786344e-06, 6.946133009185864e-06,...","[-1.8861402253599493e-05, -5.826087682652995e-...","[-1.500956963653611e-05, -5.045774848604877e-0..."
4,"Mar 2, 2024 at 2:50:08 PM",Bench,Anwar,26.90,5,205,7,None,13.00,22.62,...,"[0.611, 0.61, 0.611, 0.611, 0.611, 0.611, 0.61...","[0.001, 0.0, 0.0, -0.0, -0.001, -0.001, -0.002...","[-0.0, 0.001, 0.001, 0.002, 0.003, 0.005, 0.00...","[100, 100, 100, 100, 100, 100, 100, 100, 100, ...","[-0.00172886797983311, -0.004979273183221311, ...","[-0.0005459052611392812, -0.000848742559796294...","[-0.00023878220378011446, -0.00083529769310464...","[-3.35407058152721e-05, -0.0001035187551256329...","[-6.97323910467788e-06, -2.1634629147064263e-0...","[-5.37039948442379e-06, -1.7894732117329768e-0..."
5,"Jun 4, 2024 at 4:32:13 PM",Bench,Anwar,36.65,6,205,8,None,16.88,30.08,...,"[0.218, 0.218, 0.218, 0.217, 0.217, 0.216, 0.2...","[-0.017, -0.017, -0.018, -0.018, -0.018, -0.01...","[0.002, 0.002, 0.002, 0.001, 0.001, 0.001, 0.0...","[89, 89, 89, 89, 89, 89, 89, 89, 89, 89, 89, 8...","[-0.0039493686193578055, -0.008184311621794938...","[-0.0008618083887122424, 0.0002007887673182201...","[-0.0005330117004746367, -0.001922872767692140...","[-6.066840120576372e-05, -0.000156814858390224...","[-3.3050981069701115e-06, 1.5128931508776925e-...","[-1.2279422340833886e-05, -3.651691510035014e-..."
6,"Mar 17, 2024 at 1:43:08 PM",Bench,Anwar,36.49,5,225,9,None,17.11,30.2,...,"[0.313, 0.315, 0.317, 0.318, 0.319, 0.319, 0.3...","[0.074, 0.074, 0.073, 0.073, 0.072, 0.072, 0.0...","[-0.012, -0.012, -0.013, -0.013, -0.012, -0.01...","[98, 98, 98, 98, 98, 98, 98, 98, 98,

## Define feature extraction

In [82]:
from scipy.stats import skew, kurtosis

In [83]:
def safe_float_convert(value):
    try:
        return float(value)
    except ValueError:
        return np.nan

def parse_column(column_data):
    return np.array([safe_float_convert(x) for x in column_data.split(',')])

def extract_fft_features(window):
    fft = np.fft.fft(window)
    magnitudes = np.abs(fft)
    freqs = np.fft.fftfreq(len(fft), d=1/SAMPLE_RATE)
    magnitudes = magnitudes[freqs >= 0]
    top_5_freq_indices = np.argsort(magnitudes)[::-1][:5]
    top_5_magnitudes = magnitudes[top_5_freq_indices]

    return top_5_magnitudes.tolist()

def extract_features(window):
    mean = np.mean(window)
    std = np.std(window)
    abs_dev = np.mean(np.abs(window - np.mean(window)))
    median = np.median(window)
    median_abs_dev = np.median(np.abs(window - median))
    iqr = np.percentile(window, 75) - np.percentile(window, 25)
    neg_count = np.sum(window < 0)
    pos_count = np.sum(window > 0)
    above_mean_count = np.sum(window > mean)
    num_peaks = len(find_peaks(window)[0])
    energy = np.sum(window ** 2)
    skewness = skew(window)
    kurt = kurtosis(window)
    avg_resultant_acc = np.mean(np.sqrt(window ** 2))
    sma = np.sum(np.abs(window))
    
    features = [mean,
                std,
                abs_dev,
                median,
                median_abs_dev,
                iqr,
                neg_count,
                pos_count,
                above_mean_count,
                num_peaks,
                energy,
                skewness,
                kurt,
                avg_resultant_acc,
                sma]
    
    features += extract_fft_features(window)    
    return features

## Train a exercise classifier

In [84]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [86]:
df['AccXFeatures'] = df['AccX'].apply(lambda x: extract_features(x))
df['AccYFeatures'] = df['AccY'].apply(lambda x: extract_features(x))
df['AccZFeatures'] = df['AccZ'].apply(lambda x: extract_features(x))
df['VelXFeatures'] = df['VelX'].apply(lambda x: extract_features(x))
df['VelYFeatures'] = df['VelY'].apply(lambda x: extract_features(x))
df['VelZFeatures'] = df['VelZ'].apply(lambda x: extract_features(x))
df['PosXFeatures'] = df['PosX'].apply(lambda x: extract_features(x))
df['PosYFeatures'] = df['PosY'].apply(lambda x: extract_features(x))
df['PosZFeatures'] = df['PosZ'].apply(lambda x: extract_features(x))
df['PitchFeatures'] = df['Pitch'].apply(lambda x: extract_features(x))
df['RollFeatures'] = df['Roll'].apply(lambda x: extract_features(x))
df['YawFeatures'] = df['Yaw'].apply(lambda x: extract_features(x))

for col in ['AccXFeatures', 'AccYFeatures', 'AccZFeatures', 'VelXFeatures', 'VelYFeatures', 'VelZFeatures', 'PosXFeatures', 'PosYFeatures', 'PosZFeatures', 'PitchFeatures', 'RollFeatures', 'YawFeatures']:
    if isinstance(df[col].iloc[0], list):
        df[col] = df[col].apply(lambda x: x[0] if isinstance(x, list) else x)

accx_features = np.array(df['AccXFeatures'].tolist())
accy_features = np.array(df['AccYFeatures'].tolist())
accz_features = np.array(df['AccZFeatures'].tolist())
velx_features = np.array(df['VelXFeatures'].tolist())
vely_features = np.array(df['VelYFeatures'].tolist())
velz_features = np.array(df['VelZFeatures'].tolist())
posx_features = np.array(df['PosXFeatures'].tolist())
posy_features = np.array(df['PosYFeatures'].tolist())
posz_features = np.array(df['PosZFeatures'].tolist())
pitch_features = np.array(df['PitchFeatures'].tolist())
roll_features = np.array(df['RollFeatures'].tolist())
yaw_features = np.array(df['YawFeatures'].tolist())

df


Training accuracy:  1.0
Testing accuracy:  1.0
              precision    recall  f1-score   support

       Bench       1.00      1.00      1.00         9

    accuracy                           1.00         9
   macro avg       1.00      1.00      1.00         9
weighted avg       1.00      1.00      1.00         9



In [ ]:
X = df[['AccXFeatures', 'AccYFeatures', 'AccZFeatures', 'VelXFeatures', 'VelYFeatures', 'VelZFeatures', 'PosXFeatures', 'PosYFeatures', 'PosZFeatures', 'PitchFeatures', 'RollFeatures', 'YawFeatures']]
y = df['Exercise']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

clf = RandomForestClassifier(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, random_state=RANDOM_STATE)
clf.fit(X_train, y_train)

print("Training accuracy: ", clf.score(X_train, y_train))
print("Testing accuracy: ", clf.score(X_test, y_test))

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0))

## Train a predictor for whether is in rep state or non rep state

 For entries with a RepStartTime and RepEndTime
 - Extract windows for each axes
 - Label window with 0 or 1 depending on whether it is between RepStartTime and RepEndTime
 - Create a classifier for this data

### Extract windows

In [88]:
## Train a predictor for whether is in rep state or non rep state

# For entries with a RepStartTime and RepEndTime
# - Extract windows for each axes
# - Label window with 0 or 1 depending on whether it is between RepStartTime and RepEndTime
# - Create a classifier for this data

def extract_windows(data, window_size, overlap):
    windows = []
    for i in range(0, len(data) - window_size, window_size - overlap):
        windows.append(data[i:i+window_size])
    return np.array(windows)

def extract_window_features(window):
    features = []
    for i in range(len(window)):
        features.append(extract_features(window[i]))
    return features


In [89]:
df['AccXWindows'] = df.apply(lambda x: extract_windows(x['AccX'], WINDOW_SIZE, STEP_SIZE), axis=1)
df['AccYWindows'] = df.apply(lambda x: extract_windows(x['AccY'], WINDOW_SIZE, STEP_SIZE), axis=1)
df['AccZWindows'] = df.apply(lambda x: extract_windows(x['AccZ'], WINDOW_SIZE, STEP_SIZE), axis=1)
df['VelXWindows'] = df.apply(lambda x: extract_windows(x['VelX'], WINDOW_SIZE, STEP_SIZE), axis=1)
df['VelYWindows'] = df.apply(lambda x: extract_windows(x['VelY'], WINDOW_SIZE, STEP_SIZE), axis=1)
df['VelZWindows'] = df.apply(lambda x: extract_windows(x['VelZ'], WINDOW_SIZE, STEP_SIZE), axis=1)
df['PosXWindows'] = df.apply(lambda x: extract_windows(x['PosX'], WINDOW_SIZE, STEP_SIZE), axis=1)
df['PosYWindows'] = df.apply(lambda x: extract_windows(x['PosY'], WINDOW_SIZE, STEP_SIZE), axis=1)
df['PosZWindows'] = df.apply(lambda x: extract_windows(x['PosZ'], WINDOW_SIZE, STEP_SIZE), axis=1)
df['PitchWindows'] = df.apply(lambda x: extract_windows(x['Pitch'], WINDOW_SIZE, STEP_SIZE), axis=1)
df['RollWindows'] = df.apply(lambda x: extract_windows(x['Roll'], WINDOW_SIZE, STEP_SIZE), axis=1)
df['YawWindows'] = df.apply(lambda x: extract_windows(x['Yaw'], WINDOW_SIZE, STEP_SIZE), axis=1)

df['AccXWindowFeatures'] = df['AccXWindows'].apply(lambda x: extract_window_features(x))
df['AccYWindowFeatures'] = df['AccYWindows'].apply(lambda x: extract_window_features(x))
df['AccZWindowFeatures'] = df['AccZWindows'].apply(lambda x: extract_window_features(x))
df['VelXWindowFeatures'] = df['VelXWindows'].apply(lambda x: extract_window_features(x))
df['VelYWindowFeatures'] = df['VelYWindows'].apply(lambda x: extract_window_features(x))
df['VelZWindowFeatures'] = df['VelZWindows'].apply(lambda x: extract_window_features(x))
df['PosXWindowFeatures'] = df['PosXWindows'].apply(lambda x: extract_window_features(x))
df['PosYWindowFeatures'] = df['PosYWindows'].apply(lambda x: extract_window_features(x))
df['PosZWindowFeatures'] = df['PosZWindows'].apply(lambda x: extract_window_features(x))
df['PitchWindowFeatures'] = df['PitchWindows'].apply(lambda x: extract_window_features(x))
df['RollWindowFeatures'] = df['RollWindows'].apply(lambda x: extract_window_features(x))
df['YawWindowFeatures'] = df['YawWindows'].apply(lambda x: extract_window_features(x))

df

,Date,Exercise,Lifter,WorkoutTime,Reps,Weight,Intensity,Notes,StartRepTime,EndRepTime,...,AccZWindowFeatures,VelXWindowFeatures,VelYWindowFeatures,VelZWindowFeatures,PosXWindowFeatures,PosYWindowFeatures,PosZWindowFeatures,PitchWindowFeatures,RollWindowFeatures,YawWindowFeatures
0,"Apr 27, 2024 at 10:41:29 AM",Bench,Anwar,35.35,6,205,9,None,13.44,27.93,...,"[[0.12641470531101925, 0.7816404723942529, 0.5...","[[0.10881467661861953, 0.21914133954178608, 0....","[[-0.1319753779856032, 0.1618459840410315, 0.1...","[[0.10104759309035469, 0.12292886458033951, 0....","[[0.02288252207225806, 0.05204364286550529, 0....","[[-0.06828153058665257, 0.058028969497788005, ...","[[0.021273453699936936, 0.036774292819226305, ...","[[-0.19649999999999998, 0.5931789864787862, 0....","[[-0.11749000000000001, 0.16218936432454503, 0...","[[0.30466, 0.30962584582040303, 0.285912799999..."
1,"Mar 6, 2024 at 7:00:55 AM",Bench,Anwar,35.75,4,240,10,None,15.04,28.92,...,"[[0.33832866013288465, 1.6101891275554787, 1.4...","[[0.022588556600392537, 0.13884784526959346, 0...","[[-0.4818591161570933, 0.3999216288507775, 0.3...","[[-0.1524066071568525, 0.24769373614822487, 0....","[[-0.02312460008962877, 0.020419631855180313, ...","[[-0.1329517584957437, 0.14543250170618674, 0....","[[-0.09654330183876755, 0.07576007360742512, 0...","[[-0.58213, 0.24629533714628055, 0.2073604, -0...","[[0.3370100000000001, 0.3918107832870351, 0.29...","[[0.72996, 0.7653667999070772, 0.6146271999999..."
2,"Mar 6, 2024 at 6:53:59 AM",Bench,Anwar,29.18,5,225,8,None,13.53,24.54,...,"[[1.0422830453762297, 1.1387353921342265, 0.96...","[[0.3963764314758479, 0.5103894738005159, 0.47...","[[-0.4736032830122279, 0.2633623822039388, 0.2...","[[0.46852661298490433, 0.34354973357916324, 0....","[[0.06433974128197148, 0.13420516992338766, 0....","[[-0.1964987113716328, 0.16591325917447733, 0....","[[0.1413509465450143, 0.14280745596550246, 0.1...","[[-0.27898, 0.5263652340343158, 0.477720799999...","[[0.38741, 0.5854182111106555, 0.5266694000000...","[[1.2048800000000002, 1.2243440552393758, 1.10..."
3,"Mar 6, 2024 at 6:50:48 AM",Bench,Anwar,27.94,5,185,5,None,13.95,23.43,...,"[[-0.211329714193789, 0.5938076022221245, 0.40...","[[0.15124958804405586, 0.19137480928790765, 0....","[[-0.21815382902917513, 0.15664553927202415, 0...","[[-0.1568514108305436, 0.1038830218183456, 0.0...","[[0.028338972433379656, 0.04618151903276008, 0...","[[-0.06661938653783225, 0.06707523938262253, 0...","[[-0.05311221769230683, 0.05124890671243611, 0...","[[-0.35761000000000004, 0.1091510783272433, 0....","[[0.1899, 0.13116306644783812, 0.1163059999999...","[[0.23318999999999998, 0.20867844617976242, 0...."
4,"Mar 2, 2024 at 2:50:08 PM",Bench,Anwar,26.90,5,205,7,None,13.00,22.62,...,"[[-0.07014234789777964, 0.5913702881959929, 0....","[[-0.030257920226447122, 0.07240997123644521, ...","[[-0.04898282656993937, 0.037752355277824654, ...","[[0.02911485676719622, 0.06503129120294049, 0....","[[-0.008166115094484575, 0.00681426461818763, ...","[[-0.021207926475698198, 0.01788418107293271, ...","[[0.009335511792077495, 0.015041759683225523, ...","[[0.56269, 0.05487434646535665, 0.048593399999...","[[-0.08189, 0.07628314296094517, 0.07043680000...","[[0.15032, 0.1516058626834728, 0.1381711999999..."
5,"Jun 4, 2024 at 4:32:13 PM",Bench,Anwar,36.65,6,205,8,None,16.88,30.08,...,"[[0.03603655521190381, 1.3227295657307248, 0.8...","[[0.11205005958443044, 0.302772963652854, 0.26...","[[0.05255918994698911, 0.22877404611128532, 0....","[[0.13582490231963987, 0.13136219934559132, 0....","[[0.056900935160182825, 0.06821548536288331, 0...","[[-0.014282276875090634, 0.02558032756057549, ...","[[0.042702055176147564, 0.04943315581221595, 0...","[[-0.33201, 0.4930592559723425, 0.471450200000...","[[-0.03086, 0.07190382743637504, 0.05548919999...","[[0.32230000000000003, 0.30525269859577, 0.282..."
6,"Mar 17, 2024 at 1:43:08 PM",Bench,Anwar,36.49,5,225,9,None,17.11,30.2,...,"[[0.5595327969541914, 0.5793826828020393, 0.50...","[[0.2

### Extract labels

In [115]:
def calculate_sample_index(time, sample_rate):
    return int(float(time)* float(sample_rate))

def label_data(data, start_time, end_time, sample_rate):    
    start_index = calculate_sample_index(start_time, sample_rate)
    end_index = calculate_sample_index(end_time, sample_rate)
    
    labels = np.zeros(len(data))
    labels[start_index:end_index] = 1
    
    return labels

def label_windows(data, labels, window_size):
    window_labels = []
    for i in range(0, len(data) - window_size, window_size):
        window_labels.append(1 if np.sum(labels[i:i+window_size]) > window_size/2 else 0)
    return np.array(window_labels)

In [123]:
df['Labels'] = df.apply(lambda x: label_data(x['Counter'], x['StartRepTime'], x['EndRepTime'], SAMPLE_RATE), axis=1)
df['WindowLabels'] = df.apply(lambda x: label_windows(x['Counter'], x['Labels'], WINDOW_SIZE), axis=1)